# MRAC??L1?????: ??????????

MRAC??????????????????????????????L1?????????????????

???:

1. ?Level??????????????
2. ??????????
3. ????????????
4. ?????????????????????

????:

- 1?SISO MRAC????????
- ?????????Lyapunov??
- ????????$e \in L_2$?Barbalat?PE??
- ????$\sigma$-modification?e-modification?projection
- ???MRAC??????
- ??????? $T_s\Gamma$ ???
- L1??????????????????????????


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["font.size"] = 11


def step_signal(t):
    return np.ones_like(t)


def rich_signal(t):
    return 0.8*np.sin(0.7*t) + 0.35*np.sin(2.1*t) + 0.25*np.sign(np.sin(0.19*t))


def plot_series(t, data, title, ylabel="value"):
    plt.figure()
    for label, y in data.items():
        plt.plot(t, y, label=label)
    plt.title(title)
    plt.xlabel("time [s]")
    plt.ylabel(ylabel)
    plt.legend()
    plt.show()


def moving_window_min_eig(phi, dt, window_s):
    nwin = max(2, int(window_s/dt))
    vals = np.full(phi.shape[0], np.nan)
    for k in range(nwin, phi.shape[0]):
        gram = phi[k-nwin:k].T @ phi[k-nwin:k] * dt
        vals[k] = np.linalg.eigvalsh(gram).min()
    return vals


## Level 1: 1?SISO MRAC????????

??????????????????

$$\dot{x}=a x+b u, \qquad \dot{x}_m=a_m x_m+b_m r$$

???? $a_m<0$?$b>0$ ??????????

$$u=\hat{\theta}_1 x+\hat{\theta}_2 r$$

?????????????????????????????????

### ?????

1. ??????? $\theta_1^*,\theta_2^*$ ??????
2. $b$ ??????????????????????????
3. $e=x-x_m$ ????????$\hat{\theta}=\theta^*$ ?????????

### ?????

$$\dot{x}=(a+b\theta_1)x+b\theta_2 r$$

??????????

$$\theta_1^*=\frac{a_m-a}{b},\qquad \theta_2^*=\frac{b_m}{b}$$


## Level 2: ????????

$$e=x-x_m,\qquad \tilde{\theta}=\hat{\theta}-\theta^*,\qquad \phi=\begin{bmatrix}x\\r\end{bmatrix}$$

?????

### ?????

1. $\dot{e}$ ? $e$ ? $\tilde{\theta}$ ?????????
2. ??????????????????????????

### ??

$$\dot{e}=a_m e + b\tilde{\theta}^T\phi$$

????????????Lyapunov???????????????????


## Level 3: Lyapunov????????

$$V=\frac{1}{2}e^2+\frac{b}{2\gamma}\tilde{\theta}^T\tilde{\theta}$$

??????$\theta^*$ ?????? $\dot{\tilde{\theta}}=\dot{\hat{\theta}}$ ???

### ?????

1. $\dot{V}$ ??????????????
2. ??????????????
3. $\dot{V}\le 0$ ??????????????????????

### ??

$$\dot{V}=a_m e^2+b\tilde{\theta}^T\left(e\phi+\frac{1}{\gamma}\dot{\hat{\theta}}\right)$$

?????

$$\dot{\hat{\theta}}=-\gamma e\phi$$

????

$$\dot{V}=a_m e^2\le 0$$

??????


In [ ]:
def simulate_continuous_mrac(
    T=25.0, dt=0.001, a=-0.4, b=1.4, am=-2.0, bm=2.0,
    gamma=5.0, signal="rich", normalized=False, sigma=0.0,
    e_mod=False, projection=None, sign_b=1.0, theta0=(0.0, 0.0)
):
    n = int(T/dt) + 1
    t = np.linspace(0, T, n)
    r = step_signal(t) if signal == "step" else rich_signal(t)
    x = np.zeros(n)
    xm = np.zeros(n)
    theta = np.zeros((n, 2))
    theta[0] = theta0
    u = np.zeros(n)
    e = np.zeros(n)
    theta_star = np.array([(am-a)/b, bm/b])

    for k in range(n-1):
        phi = np.array([x[k], r[k]])
        u[k] = theta[k] @ phi
        e[k] = x[k] - xm[k]
        denom = 1.0 + phi @ phi if normalized else 1.0
        leak = sigma * (abs(e[k]) if e_mod else 1.0)
        dtheta = -gamma * sign_b * e[k] * phi / denom - leak * theta[k]
        next_theta = theta[k] + dt*dtheta
        if projection is not None:
            lo, hi = projection
            next_theta = np.clip(next_theta, lo, hi)
        theta[k+1] = next_theta
        x[k+1] = x[k] + dt*(a*x[k] + b*u[k])
        xm[k+1] = xm[k] + dt*(am*xm[k] + bm*r[k])

    u[-1] = theta[-1] @ np.array([x[-1], r[-1]])
    e[-1] = x[-1] - xm[-1]
    return {"t": t, "r": r, "x": x, "xm": xm, "e": e, "u": u, "theta": theta, "theta_star": theta_star}

res = simulate_continuous_mrac(gamma=5.0, signal="rich")
plot_series(res["t"], {"x": res["x"], "xm": res["xm"]}, "Continuous-time MRAC: output tracking")
plot_series(res["t"], {"e": res["e"]}, "Tracking error")
plot_series(res["t"], {"theta1_hat": res["theta"][:,0], "theta2_hat": res["theta"][:,1],
                       "theta1_star": np.full_like(res["t"], res["theta_star"][0]),
                       "theta2_star": np.full_like(res["t"], res["theta_star"][1])}, "Parameter estimates")


## Level 4: $e\in L_2$?Barbalat?PE??

$\dot{V}=a_m e^2$ ?? $a_m<0$ ????$V(t)\le V(0)$ ??????

$$\int_0^\infty e^2(t)dt < \infty$$

??? $e\in L_2$ ??????

????????????????

- ??????? $e(t)\to0$ ????
- ??? $\tilde{\theta}(t)\to0$ ?????????
- ??????????????? $\phi$ ?PE?????

PE??????:

$$\exists T>0,\alpha>0\quad
\int_t^{t+T}\phi(\tau)\phi^T(\tau)d\tau \ge \alpha I$$

### ?????

1. ?????????????????????????????????
2. ???????????????????????


In [ ]:
for sig in ["step", "rich"]:
    rr = simulate_continuous_mrac(gamma=5.0, signal=sig, T=35.0)
    phi = np.column_stack([rr["x"], rr["r"]])
    pe = moving_window_min_eig(phi, dt=0.001, window_s=4.0)
    plot_series(rr["t"], {"theta1_hat": rr["theta"][:,0], "theta2_hat": rr["theta"][:,1]}, f"Parameter estimates with {sig} input")
    plot_series(rr["t"], {"min eig of windowed Gramian": pe}, f"PE indicator with {sig} input", ylabel="min eigenvalue")


## Level 5: ???????????????

$b$ ?????????????? $\operatorname{sgn}(b)$ ?????????????????

$$\dot{\hat{\theta}}=-\gamma\operatorname{sgn}(b)e\phi$$

??????

### ?????

1. ?????????????????????????
2. $b$ ???????????????????????????????????????


In [ ]:
ok = simulate_continuous_mrac(gamma=2.0, signal="rich", sign_b=1.0, T=15.0)
bad = simulate_continuous_mrac(gamma=2.0, signal="rich", sign_b=-1.0, T=15.0)
plot_series(ok["t"], {"correct sign e": ok["e"], "wrong sign e": bad["e"]}, "Input-gain sign matters")
plot_series(ok["t"], {"correct sign u": ok["u"], "wrong sign u": bad["u"]}, "Control input with correct/wrong sign")


## Level 6: ????$\sigma$-modification?e-modification?projection

??????

$$\dot{\hat{\theta}}=-\Gamma\phi e$$

??$\phi$ ????????????????????????????????

???:

$$\dot{\hat{\theta}}=-\Gamma\frac{\phi e}{1+\phi^T\phi}$$

$\sigma$-modification:

$$\dot{\hat{\theta}}=-\Gamma\phi e-\sigma\hat{\theta}$$

e-modification:

$$\dot{\hat{\theta}}=-\Gamma\phi e-\sigma |e|\hat{\theta}$$

projection:

$$\hat{\theta}\in\Omega$$

### ?????

1. ???????????????????????????
2. $\sigma$-modification??????????????????????????????????
3. projection???????????????????


In [ ]:
plain = simulate_continuous_mrac(gamma=25.0, signal="rich", T=20.0)
norm = simulate_continuous_mrac(gamma=25.0, signal="rich", T=20.0, normalized=True)
sig = simulate_continuous_mrac(gamma=25.0, signal="rich", T=20.0, sigma=0.6)
emod = simulate_continuous_mrac(gamma=25.0, signal="rich", T=20.0, sigma=0.6, e_mod=True)
proj = simulate_continuous_mrac(gamma=25.0, signal="rich", T=20.0, projection=(-3.0, 3.0))
plot_series(plain["t"], {"plain": plain["e"], "normalized": norm["e"], "sigma": sig["e"], "e-mod": emod["e"], "projection": proj["e"]}, "Robust/adaptive modifications: error")
plot_series(plain["t"], {"plain": plain["u"], "normalized": norm["u"], "sigma": sig["u"], "e-mod": emod["u"], "projection": proj["u"]}, "Robust/adaptive modifications: control input")


## Level 7: ??????? $T_s\Gamma$

?????????????????????????? $T_s$ ????????????Euler????

$$\hat{\theta}_{k+1}=\hat{\theta}_k-T_s\Gamma\phi_k e_k$$

??????????? $\Gamma$ ??????????? $T_s\Gamma$ ????????

Tustin?????????????????????????????????????????????????????????????Euler??? $T_s\Gamma$ ??????????????????

### ?????

1. $T_s$ ???????? $\Gamma$ ???????????????
2. Tustin????????????Euler???????????????????


In [ ]:
def simulate_discrete_mrac(T=25.0, Ts=0.02, a=-0.4, b=1.4, am=-2.0, bm=2.0, gamma=5.0, signal="rich"):
    n = int(T/Ts) + 1
    t = np.linspace(0, T, n)
    r = step_signal(t) if signal == "step" else rich_signal(t)
    x = np.zeros(n)
    xm = np.zeros(n)
    theta = np.zeros((n, 2))
    u = np.zeros(n)
    e = np.zeros(n)
    for k in range(n-1):
        phi = np.array([x[k], r[k]])
        u[k] = theta[k] @ phi
        e[k] = x[k] - xm[k]
        theta[k+1] = theta[k] - Ts*gamma*e[k]*phi
        x[k+1] = x[k] + Ts*(a*x[k] + b*u[k])
        xm[k+1] = xm[k] + Ts*(am*xm[k] + bm*r[k])
    u[-1] = theta[-1] @ np.array([x[-1], r[-1]])
    e[-1] = x[-1] - xm[-1]
    return {"t": t, "x": x, "xm": xm, "e": e, "u": u, "theta": theta}

cases = [(0.002, 5.0), (0.02, 5.0), (0.02, 60.0), (0.08, 60.0)]
plt.figure(figsize=(10, 4))
for Ts, gamma in cases:
    rr = simulate_discrete_mrac(Ts=Ts, gamma=gamma, T=15.0)
    plt.plot(rr["t"], rr["e"], label=f"Ts={Ts}, gamma={gamma}, Ts*gamma={Ts*gamma:.2f}")
plt.title("Discrete MRAC: Ts*Gamma sensitivity")
plt.xlabel("time [s]")
plt.ylabel("e")
plt.legend()
plt.show()


## Level 8: L1?????????

??MRAC???????????????????????

$$u=\hat{\theta}^T\phi$$

?????$\Gamma$ ????????????????????????????????????????????????????????????????????????

L1????????????????????

- ?????????????
- ??????????????????????
- ??????????????????

?????????????? $\hat{\sigma}$ ????

$$u_{ad}(s)=-C(s)\hat{\sigma}(s),\qquad C(s)=\frac{\omega}{s+\omega}$$

????????????????

### ?????

1. MRAC? $\Gamma$ ???????????????????????????
2. L1????????????????????????????
3. ?????? $\omega$ ??????????MRAC??????????


In [ ]:
def lowpass_filter(signal, t, omega):
    y = np.zeros_like(signal)
    dt = t[1] - t[0]
    for k in range(len(signal)-1):
        y[k+1] = y[k] + dt*omega*(signal[k] - y[k])
    return y

fast = simulate_continuous_mrac(gamma=80.0, signal="rich", T=12.0, normalized=True)
raw_ad = fast["u"]
filtered_2 = lowpass_filter(raw_ad, fast["t"], omega=2.0)
filtered_10 = lowpass_filter(raw_ad, fast["t"], omega=10.0)
plot_series(fast["t"], {"raw MRAC-like adaptive input": raw_ad, "LPF omega=2": filtered_2, "LPF omega=10": filtered_10}, "L1 idea: fast estimate, filtered control channel", ylabel="input")


## Level 9: ???MRAC??????

??????????????MRAC???????

$$\dot{x}=Ax+Bu,\qquad \dot{x}_m=A_mx_m+B_mr$$

???????? $e=x-x_m$ ?????Lyapunov???

$$A_m^TP+PA_m=-Q$$

????

$$V=e^TPe+\tilde{\theta}^T\Gamma^{-1}\tilde{\theta}$$

?????????????????

$$2e^TPB\tilde{\theta}^T\phi$$

??????????????

$$\dot{\hat{\theta}}=-\Gamma\phi e^TPB$$

????????

### ?????

1. 1??? $e$ ??????????????????????
2. ?? $P$ ??????????
3. ?????????MRAC??????????????????


## Level 10: ?????

?????????????

1. `signal="step"` ? `signal="rich"` ?PE????????
2. `sign_b=-1.0` ???????????????????????????
3. `gamma` ????????????????????
4. `sigma` ? `e_mod=True` ????????????????
5. `simulate_discrete_mrac` ? `Ts*gamma` ??????????
6. L1????????? `omega` ???????????????????????

????????:

- MRAC???????????????????????????
- ??????????????????????? $T_s\Gamma$ ????
- L1?????????????????????????????????????
